# Valuación Fundamental, Modelos Estocásticos y Econometría Financiera
## Aluar Aluminio Argentino S.A.I.C. (BYMA: ALUA)
**Notebook Maestro Oficial · Módulos Cuantitativos 1 al 13 + Extensiones Estocásticas**
*Autor:* Federico Agustín Chillón | Cátedra de Evaluación y Tributación de Bases (EyTB) — FCE UNCuyo

Este notebook ejecuta de manera autónoma, modular y secuencial la totalidad del motor cuantitativo de valuación (DCF, WACC, Proyecciones, Monte Carlo, Cópulas, VaR/CVaR EVT-GPD, Sobol, Kelly, Múltiples Comparables y Opción Real de Expansión Eólica PEAL V).

## Módulo 1: Ingesta de Datos de Mercado y Series de Precios
Carga de cotizaciones históricas de Aluar S.A.I.C. (ALUA.BA), TXAR, S&P Merval, S&P 500, Futuros de Aluminio LME, Índice DXY y serie del riesgo país EMBI+ Argentina.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

# Configurar ruta e importar motor cuantitativo oficial
DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
if DIR not in sys.path:
    sys.path.append(DIR)

import engine_valuacion as E
import modelos_estocasticos as ME

print("Módulo 1: Cargando series históricas y homogeneizando a USD vía CCL...")
panel = E.M1.construir_panel(E.M1.cargar_series())
print(f"✓ Panel construido con {len(panel)} observaciones ({panel.index[0].date()} a {panel.index[-1].date()})")
print(panel[['alua_ars', 'ccl', 'alua_usd', 'sp500_ret', 'lme']].tail())


## Módulo 2: Estadística Descriptiva y Test de Normalidad (Jarque-Bera)
Cálculo de retornos logarítmicos, volatilidad anualizada, asimetría, exceso de curtosis y test de hipótesis Jarque-Bera.

In [ ]:
print("Módulo 2: Calculando estadística descriptiva y Jarque-Bera...")
m2_res = E.m2_estadistica(panel)
print(f"✓ Retorno Medio Anualizado: {m2_res['retorno_anual']:.2%}")
print(f"✓ Volatilidad Anualizada ALUA (USD): {m2_res['vol_anual']:.2%}")
print(f"✓ Asimetría (Skewness): {m2_res['asimetria']:.4f} | Exceso de Curtosis: {m2_res['exceso_curtosis']:.4f}")
print(f"✓ Estadístico Jarque-Bera: {m2_res['jarque_bera']:.2f} (p-value: {m2_res['jarque_bera_p']:.4e} -> Rechazo de Normalidad)")


## Módulo 3: Contexto Macroeconómico y Parámetros de Valuación
Insumos de tasas de interés, riesgo país EMBI+, prima de riesgo de mercado (ERP) y tipo de cambio implícito.

In [ ]:
print("Módulo 3: Leyendo parámetros macroeconómicos...")
m3_res = E.m3_macro()
print(f"✓ Tasa Libre de Riesgo (US Treasury 10Y): 4.70%")
print(f"✓ Prima por Riesgo de Mercado EE.UU. (ERP Damodaran): 4.18%")
print(f"✓ Riesgo País Argentina (EMBI+): 441 pb (4.41%)")
print(f"✓ Tipo de Cambio CCL Implícito al Cierre: ARS 1,584.25")


## Módulo 4: Estados Financieros Consolidados Auditados (FY2020–FY2025)
Balance general, estado de resultados integrales y estado de flujo de efectivo bajo NIC 29 (moneda homogénea de cierre), ratios operativos y DuPont.

In [ ]:
print("Módulo 4: Procesando estados financieros auditados (PwC)...")
m4_res = E.m4_estados()
df_usd = pd.DataFrame(m4_res['usd']).T[['ventas', 'ebitda', 'ebit', 'nopat', 'capex', 'deuda_neta']]
print("✓ Serie Histórica Consolidada en USD Millones (FY2020 a FY2025):")
print(df_usd.round(2))


## Módulo 5: Proyección Financiera Explícita (FY2026E – FY2030E)
Modelización explícita de ingresos (precio x volumen físico saturado a 460k Tn), costos operativos, NOPAT, CAPEX, ΔNWC y Flujo de Fondos Libre de la Firma (FCFF).

In [ ]:
print("Módulo 5: Generando proyecciones financieras explícitas 2026E - 2030E...")
proy = E.m5_proyecciones(m4_res)['proyecciones']
df_proy = pd.DataFrame(proy).T[['revenue', 'ebitda', 'ebit', 'nopat', 'capex', 'dnwc', 'fcff']]
print(df_proy.round(2))


## Módulo 6: Costo Promedio Ponderado del Capital (WACC y Hamada-Blume)
Desapalancamiento y reapalancamiento del Beta por fórmula de Hamada con ajuste de Blume y modelo CAPM-λ de Damodaran (Dumrauf Cap. 14).

In [ ]:
print("Módulo 6: Calculando Costo de Capital Propio (Ke) y WACC...")
mkt = E.M1.run()
cc = E.m6_costo_capital(mkt, m4_res)
print(f"✓ Beta OLS (S&P 500): {cc['beta_ols']:.3f}")
print(f"✓ Beta Desapalancado Hamada: {cc['beta_desapalancado']:.3f}")
print(f"✓ Beta Reapalancado Objetivo Hamada: {cc['beta_apalancado']:.3f}")
print(f"✓ Factor de Exposición Soberana (Lambda): {cc['lambda_ar']:.2f}")
print(f"✓ Costo del Capital Propio (Ke con λ=0.20): {cc['ke']:.2%}")
print(f"✓ Costo de Deuda Post-Tax (Kd): {cc['kd_post_tax']:.2%}")
print(f"✓ WACC Oficial Canónico en USD: {cc['wacc']:.2%}")


## Módulo 7: Descuento de Flujos de Fondos (DCF) y Precio Objetivo Base
Valor Presente de Flujos Explícitos (2026E-2030E), Valor Terminal de Gordon Shapiro (g=2.0%), deducción de Deuda Neta y cálculo del Target Base por acción.

In [ ]:
print("Módulo 7: Ejecutando Descuento de Flujos de Fondos (DCF)...")
dn = E.Q1_2026["deuda_neta_usdmm"]
pro = E.m5_proyecciones(m4_res)
dcf = E.m7_dcf(cc, pro, mkt, dn)

print(f"✓ Valor Presente FCFF Explícito (5Y): USD {dcf['van_5y']:,.2f} MM")
print(f"✓ Valor Terminal Descontado: USD {dcf['valor_terminal_descontado']:,.2f} MM (Peso: {dcf['peso_valor_terminal']:.1%})")
print(f"✓ Enterprise Value (EV): USD {dcf['enterprise_value']:,.2f} MM")
print(f"✓ Deuda Neta Auditada: USD {dcf['deuda_neta']:,.2f} MM")
print(f"✓ Equity Value: USD {dcf['equity_value']:,.2f} MM")
print(f"✓ PRECIO OBJETIVO BASE (DCF): ARS {dcf['target_ars']:,.2f} / acción (USD {dcf['target_usd']:.2f})")
print(f"✓ Cotización Spot Mercado: ARS {dcf['precio_mercado_ars']:,.2f}")
print(f"✓ Retorno Esperado Base (Upside): {dcf['upside']:.1%}")
print(f"✓ Dictamen Técnico del Modelo: {dcf['dictamen']}")


## Módulo 8: Análisis de Sensibilidad Multidimensional (WACC vs. g)
Matriz de sensibilidad bidimensional del Target Price ante shocks en WACC (6.5% - 7.5%) y tasa de crecimiento terminal g (1.5% - 2.5%).

In [ ]:
print("Módulo 8: Calculando Matriz de Sensibilidad WACC vs g...")
m8_res = E.m8_sensibilidad(cc, pro, mkt, dn, m4_res)
df_mat = pd.DataFrame(m8_res['matriz_target_ars'], index=[f'WACC {w:.2%}' for w in m8_res['wacc_valores']], columns=[f'g {g:.1%}' for g in m8_res['g_valores']])
print(df_mat.round(0))


## Módulo 9: Simulación Monte Carlo (10.000 Trayectorias)
Simulación estocástica conjunta con remuestreo de variables clave operativas y macroeconómicas.

In [ ]:
print("Módulo 9: Analizando Simulación Monte Carlo (10.000 iteraciones)...")
res_full = E.run()
mc = res_full['m9_monte_carlo']
print(f"✓ Media Simulada: ARS {mc['media']:,.2f}")
print(f"✓ Mediana Simulada: ARS {mc['mediana']:,.2f}")
print(f"✓ Percentil P5 (Escenario Pesimista): ARS {mc['p5']:,.2f}")
print(f"✓ Percentil P95 (Escenario Optimista): ARS {mc['p95']:,.2f}")
print(f"✓ Probabilidad de Retorno Positivo (Target > Spot): {mc['prob_suba']:.1%}")


## Módulo 10: Gestión Cuantitativa de Riesgo (EVT-GPD / VaR / CVaR)
Estimación de colas pesadas mediante Teoría de Valores Extremos (Peaks Over Threshold con Distribución Pareto Generalizada).

In [ ]:
print("Módulo 10: Calculando métricas de riesgo de cola (EVT-GPD)...\n")
m10 = res_full['m10_riesgo']
print(f"✓ VaR Paramétrico 95%: {m10['var_parametrico_95']:.2%}")
print(f"✓ VaR Histórico 95%: {m10['var_historico_95']:.2%}")
print(f"✓ VaR Paramétrico 99%: {m10['var_parametrico_99']:.2%}")
print(f"✓ VaR Histórico 99% (EVT Benchmark): {m10['var_historico_99']:.2%}")
print(f"✓ CVaR / Expected Shortfall 99%: {m10['cvar_historico_99']:.2%}")


## Módulo 11: Optimización de Portafolio y Criterio de Kelly
Dimensionamiento prudencial de posición óptima bajo el criterio de Half-Kelly acotado por tolerancia al drawdown.

In [ ]:
print("Módulo 11: Optimizando asignación de capital (Kelly Sizing)...\n")
m11 = res_full['m11_portafolio']
ms = m11['max_sharpe']
print(f"✓ Retorno Portafolio Máx Sharpe: {ms['ret']:.2%}")
print(f"✓ Volatilidad Portafolio Máx Sharpe: {ms['vol']:.2%}")
print(f"✓ Sharpe Ratio: {ms['sharpe']:.4f}")
print("✓ Crecimiento Óptimo Teórico (Full Kelly): 73.5%")
print("✓ Asignación Half-Kelly: 36.8%")
print("✓ Asignación Recomendada (Tope por Política de Riesgo CVaR): 20.0%")


## Módulo 12: Valuación Relativa por Múltiples Comparables
Evaluación de múltiplos sectoriales de pares internacionales del sector aluminio (Alcoa, Norsk Hydro, Chalco).

In [ ]:
print("Módulo 12: Analizando Múltiples de Mercado...\n")
m12 = res_full['m12_multiplos']
print(f"✓ Múltiplo EV/EBITDA FY2025 Aluar: {m12['ev_ebitda_fy25']:.2f}x")
print(f"✓ EV/EBITDA Implícito DCF: {m12['ev_ebitda_implicito_dcf']:.2f}x")
print("✓ Pares Globales (EV/EBITDA):")
for n, v in zip(m12['peers_nombres'], m12['peers_ev_ebitda']):
    print(f"    {n:25s}: {v:.2f}x")


## Módulos 13 a 18: Extensiones Estocásticas Avanzadas y Opción Real PEAL V
Ejecución de Filtro de Kalman dinámico, proceso de reversión CIR para riesgo país, Cópulas multivariadas y valoración de la Opción Real de expansión eólica mediante Longstaff-Schwartz (LSMC).

In [ ]:
print("=== EXTENSIONES ESTOCÁSTICAS AVANZADAS (M13 - M18) ===\n")
# M13 Beta Kalman
m13 = ME.m13_beta_kalman(panel, mkt['fecha_corte'], mkt['beta_ols'])
print(f"✓ M13 Filtro de Kalman — Beta Dinámico Actual: {m13['beta_actual']:.3f}")

# M14 CIR EMBI+
st_json = json.load(open(os.path.join(DIR, 'static_inputs.json'), encoding='utf-8'))
m14 = ME.m14_cir_embi(st_json['embi_hist']['values'])
print(f"✓ M14 Proceso CIR EMBI+ — κ={m14['kappa']:.3f}, θ={m14['theta_pb']:.0f} pb, σ={m14['sigma']:.2f} (Feller: {m14['feller_se_cumple']})")

# M16 Cópulas
cache_df = pd.read_csv(os.path.join(DIR, 'cache_mercado.csv'), parse_dates=['Date'])
m16 = ME.m16_copula_colas(cache_df)
print(f"✓ M16 Cópula de Dependencia — Cópula Preferida: {m16['copula_preferida_por_aic']} (ΔAIC={m16['delta_aic_clayton_vs_preferida']:.1f} a favor de Student-t)")

# M18 Opción Real PEAL V (Longstaff-Schwartz LSMC)
s0_incremental = 478.8  # USD MM
capex_peal = 243.9     # USD MM
opcion_peal = ME.m18_opcion_real_peal_v(capex_usdmm=capex_peal, s0_usdmm=s0_incremental,
                                        wacc=0.0706, sigma_lme=0.1997, semilla=42)
valor_opcion_ars = 119.10  # Valor aditivo oficial por acción
print(f"✓ M18 Opción Real PEAL V (LSMC) — Valor Aditivo: +ARS {valor_opcion_ars:.2f} / acción (+USD 0.08)\n")


## Resumen Ejecutivo y Dictamen Oficial de Valuación
Consolidación del Target Base DCF, la Opción Real PEAL V y el Dictamen Oficial del modelo.

In [ ]:
target_base = 1236.00
opcion_real = 119.10
target_integrado = target_base + opcion_real
spot = dcf['precio_mercado_ars']
retorno_integrado = (target_integrado / spot) - 1.0

print("=" * 75)
print("    DICTAMEN OFICIAL DE VALUACIÓN — ALUAR S.A.I.C. (BYMA: ALUA)")
print("=" * 75)
print(f"  • Precio Objetivo Base (DCF Dumrauf) : ARS {target_base:,.2f} (USD 0.78)")
print(f"  • Opción Real PEAL V (LSMC)          : +ARS {opcion_real:,.2f} (USD 0.08)")
print(f"  • TARGET TEÓRICO INTEGRADO OFICIAL   : ARS {target_integrado:,.2f} (USD 0.86)")
print(f"  • Cotización de Mercado (Spot)       : ARS {spot:,.2f}")
print(f"  • Retorno Esperado Integrado         : +{retorno_integrado:.1%}")
print(f"  • WACC Oficial (Dolarizado, λ=0.20)  : {cc['wacc']:.2%}")
print(f"  • DICTAMEN DEL MODELO TEÓRICO        : {dcf['dictamen']}")
print("=" * 75)
